In [1]:
import pandas as pd
import os
import json

# Initialize output file
output_file = "/kaggle/working/labeled_results.jsonl"

# Initialize the file if it doesn't exist
if not os.path.exists(output_file):
    open(output_file, 'w').close()


# open(output_file, "w").close()

In [2]:
!pip install -U 'tensorflow[and-cuda]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.6/620.6 MB 2.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nccl-cu12
    Found existing installation: nvidia-nccl-cu12 2.21.5
    Uninstalling nvidia-nccl-cu12-2.21.5:
      Successfully uninstalled nvidia-nccl-cu12-2.21.5
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.4.1
    Uninstalling ml-dtypes-0.4.1:
      Successfully uninstalled ml-dtypes-0.4.1
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall: keras
    Found existing installation: keras 3.8.

In [3]:
from transformers import pipeline

In [4]:
labels = {
    "beginner": "print statements, variables, loops, functions, classes, syntax",
    "intermediate": "frameworks, debugging, APIs, libraries, data processing, workflow",
    "expert": "algorithms, neural networks, optimization, system design, research"
}

In [5]:
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [6]:
from tqdm.auto import tqdm
import math
def classify_text(data, batch_size=32, multi_class=False, save=False):
    """
    Classify one or many texts using the global `classifier`
    and global `labels` dict (label → description).
    """

    texts = data["init_message"].head(10).tolist()
    convo_hashes = data["conversation_hash"].head(10).tolist()
    row_ids = data["Unnamed: 0"].head(10).tolist()

    # Accept single string
    if isinstance(texts, str):
        texts = [texts]
        convo_hashes = ["single_convo"]
        row_ids = [0]

    label_names = list(labels.keys())
    label_descs = list(labels.values())

    results = []
    num_batches = math.ceil(len(texts) / batch_size)

    for i in tqdm(range(num_batches), desc="Classifying"):
        batch_start = i * batch_size
        batch_end = batch_start + batch_size

        batch_texts = texts[batch_start : batch_end]
        batch_hashes = convo_hashes[batch_start : batch_end]
        batch_ids = row_ids[batch_start : batch_end]

        out = classifier(
            batch_texts,
            candidate_labels=label_descs,
            hypothesis_template="This question is about {}."
        )

        if isinstance(out, dict):
            out = [out]

        # Map back from description → label key and attach metadata
        for j, r in enumerate(out):
            # Create scores dict mapping label names to their scores
            # The classifier returns labels and scores in descending order
            scores_dict = {}
            for desc, score in zip(r["labels"], r["scores"]):
                label_name = label_names[label_descs.index(desc)]
                scores_dict[label_name] = score
            
            record = {
                "row_id": int(batch_ids[j]),
                "init_message": r["sequence"],
                "conversation_hash": batch_hashes[j],
                "scores": scores_dict
            }
            results.append(record)

        if save and output_file:
            # Append each record as a JSON line
            with open(output_file, 'a') as f:
                for record in results[-len(out):]:  # Only write the current batch
                    f.write(json.dumps(record) + '\n')

    return results

In [7]:
data = pd.read_csv("/kaggle/input/wildchat-hash-init/wildchat_convohash_initmessage.csv")

In [8]:
data

,Unnamed: 0,conversation_hash,init_message
0,0,cf1267ca6b2f6fccc9c36652a00059a1,"Old age PT hx of DM, HTN, dyslipidemia His ECG..."
1,1,e98d3e74c57f9a65261df393d9124ac2,Hey there! Are you familiar with reality shift...
2,2,2e8fd255aab694b07a0be8d83cb53a7b,Hey there! Are you familiar with reality shift...
3,3,59c72510f3143025f94f75b883b026bd,i wanna you to write me terms & conditions and...
4,4,a46dca428c5be27147ab40a54ed348f8,Hey there! Are you familiar with reality shift...
...,...,...,...
48386,48386,74d7e10227ac9fac0a0bf83e96109fd1,"In Google Apps Script, create a form that sele..."
48387,48387,d6820d586a6119f31b27f4a8e9982a80,﻿\n\nThe intensity of a sound varies inversely...
48388,48388,bd19cf9c09ba550346a055adc86a6e21,create ai buy and sell indicator
48389,48389,20a297f6f17b51d385fcf0bb81038ab1,GetFromJsonAsync c# how to pass json body


In [9]:
classify_text(data, batch_size=64, save=True)

Classifying:   0%|          | 0/1 [00:00<?, ?it/s]

[{'row_id': 0,
  'init_message': 'Old age PT hx of DM, HTN, dyslipidemia His ECG I.II, aVF (MI) what is the highest risk \n\nfactor for this condition?',
  'conversation_hash': 'cf1267ca6b2f6fccc9c36652a00059a1',
  'scores': {'intermediate': 0.39865973591804504,
   'beginner': 0.31867852807044983,
   'expert': 0.2826617658138275}},
 {'row_id': 1,
  'init_message': "Hey there! Are you familiar with reality shifting? So, I’m refining a foolproof method for reality shifting and want to pick a destination. Want to help me? I’m thinking something pretty personalized. There are a few things that are required of my destination. 1. The quest. I have to have a clear overarching goal in my reality, and don’t make it too crazy. It should be more along the lines of “save the president’s daughter” or “escape this weird wacky sinister place” NOT “get an artifact that literally controls reality”. Seriously, don’t make me fetch an artifact. Don't make me fetch anything, make me DO something. 2. Babes.